# CSIS3754 – Examination Question 2 (End-of-Year 2024)
## Unsupervised Machine Learning: Water Quality Clustering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

## 2.1 Read the data file

In [ ]:
water = pd.read_csv('waterquality.csv')
print(f'Dataset loaded successfully.')
water.head()

## 2.2 Brief Summary of the Dataset

### Number of records and features

In [ ]:
print(f'Number of records : {water.shape[0]}')
print(f'Number of features: {water.shape[1]}')

### Records 1001 to 1020 (inclusive)

In [ ]:
print('Records 1001 to 1020 (inclusive):')
water.iloc[1000:1020]

### Statistical summary

In [ ]:
print('Statistical summary of all features:')
water.describe()

### Concise dataframe summary

In [ ]:
print('Concise summary (index, dtypes, columns, non-null values, memory usage):')
water.info()

### Check for missing values

In [ ]:
print('Missing values per feature:')
missing = water.isnull().sum()
missing_pct = (missing / len(water) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

## 2.3 Handle Missing Values

In [ ]:
# Date (0.21% missing): Date is used only for context and will be dropped
# in pre-processing. We drop the rows with missing dates to avoid index issues.
print(f'Rows before dropping missing Date: {len(water)}')
water = water.dropna(subset=['Date']).reset_index(drop=True)
print(f'Rows after  dropping missing Date: {len(water)}')
print(f'Missing Date remaining: {water["Date"].isnull().sum()}')

In [ ]:
# DissolvedOxygen (35.89% missing): high percentage — dropping these rows
# would lose too much data. We impute with the median, which is robust to
# the skewed distribution that environmental measurements often exhibit.
do_median = water['DissolvedOxygen (mg/L)'].median()
water['DissolvedOxygen (mg/L)'] = water['DissolvedOxygen (mg/L)'].fillna(do_median)
print(f'DissolvedOxygen: filled {851} NaN values with median ({do_median}).')
print(f'Missing remaining: {water["DissolvedOxygen (mg/L)"].isnull().sum()}')
water[['DissolvedOxygen (mg/L)']].describe()

In [ ]:
# Salinity (5.48%), pH (4.01%), WaterTemp (5.10%): moderate missing rates.
# Impute with the median for the same reason — robustness to outliers.
for col in ['Salinity (ppt)', 'pH', 'WaterTemp (C)']:
    med = water[col].median()
    water[col] = water[col].fillna(med)
    print(f'{col}: filled NaN with median ({med:.4f}). Missing remaining: {water[col].isnull().sum()}')

In [ ]:
# SecchiDepth (3.08%) and WaterDepth (2.99%): low missing rates.
# Also impute with median for consistency.
for col in ['SecchiDepth (m)', 'WaterDepth (m)']:
    med = water[col].median()
    water[col] = water[col].fillna(med)
    print(f'{col}: filled NaN with median ({med:.4f}). Missing remaining: {water[col].isnull().sum()}')

In [ ]:
print('Confirmation — total missing values remaining:')
print(water.isnull().sum())
print(f'\nTotal: {water.isnull().sum().sum()}')

## 2.4 Pre-processing

In [ ]:
print('Data types before pre-processing:')
print(water.dtypes)
print()
print('Object columns:', water.select_dtypes(include='object').columns.tolist())

In [ ]:
# Date is a string (object) with 801 unique values. It is a timestamp identifier
# and carries no direct numerical meaning for clustering. We drop it.
water = water.drop(columns=['Date'])
print('Date column dropped (non-numeric identifier not useful for clustering).')
print('Remaining columns:', water.columns.tolist())

In [ ]:
print('Confirmation — no object columns remaining:')
print(water.dtypes)
print()
obj_cols = water.select_dtypes(include='object').columns.tolist()
print(f'Object columns: {obj_cols if obj_cols else "None — all features are numeric"}')

In [ ]:
# The seven remaining features have very different scales (e.g. pH ranges 0–14
# while WaterDepth can exceed 10 m and AirTemp spans a wide range).
# K-Means uses Euclidean distance, which is sensitive to scale.
# We apply StandardScaler to normalise all features to mean=0 and std=1.
scaler = StandardScaler()
water_scaled = scaler.fit_transform(water)
water_scaled_df = pd.DataFrame(water_scaled, columns=water.columns)
print('StandardScaler applied. Scaled feature summary (mean ≈ 0, std ≈ 1):')
water_scaled_df.describe().round(4)

## 2.5 K-Means Clustering — Determine Optimal k

In [ ]:
# Elbow curve and Silhouette score side by side
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=13, n_init=10)
    km.fit(water_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(water_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[0].set_title('Elbow Method — Optimal k', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_range), silhouettes, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score — Optimal k', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_k = list(K_range)[silhouettes.index(max(silhouettes))]
print(f'Best k by silhouette score: {best_k}  (score = {max(silhouettes):.4f})')

In [ ]:
# Train K-Means with optimal k
optimal_k = best_k
kmeans = KMeans(n_clusters=optimal_k, random_state=13, n_init=10)
water['Cluster'] = kmeans.fit_predict(water_scaled)

print(f'K-Means trained with optimal k = {optimal_k}')
print()
print('Cluster labels assigned to each data point:')
print(water['Cluster'].values)
print()
print('Cluster distribution:')
print(water['Cluster'].value_counts().sort_index())

## 2.6 Principal Component Analysis (PCA)

In [ ]:
# Drop Cluster column before PCA (it was added after clustering, not a measurement)
X = water.drop(columns=['Cluster'])
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print('=== Dimensions Before and After PCA ===')
print(f'Original dataset dimensions : {X_scaled.shape}')
print(f'PCA-transformed dimensions  : {X_pca.shape}')

In [ ]:
print('=== Principal Components (Feature Loadings) ===')
pc_df = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=['PC1', 'PC2']
)
print(pc_df.round(4))
print()
print('=== Explained Variance ===')
print(f'PC1 explained variance ratio : {pca.explained_variance_ratio_[0]:.4f}  ({pca.explained_variance_ratio_[0]*100:.1f}%)')
print(f'PC2 explained variance ratio : {pca.explained_variance_ratio_[1]:.4f}  ({pca.explained_variance_ratio_[1]*100:.1f}%)')
print(f'Total explained variance     : {pca.explained_variance_ratio_.sum():.4f}  ({pca.explained_variance_ratio_.sum()*100:.1f}%)')
print(f'\nExplained variance values    : {np.round(pca.explained_variance_, 4)}')

In [ ]:
# Seaborn scatter plot of PCA components coloured by cluster
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = water['Cluster'].astype(str)

cluster_colours = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
palette = {str(i): cluster_colours[i] for i in range(optimal_k)}

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2',
    hue='Cluster', palette=palette,
    s=60, alpha=0.75, edgecolor='white', linewidth=0.4
)
plt.title(f'PCA Scatterplot — Water Quality Clusters (k={optimal_k})', fontsize=14, fontweight='bold')
plt.xlabel(f'Principal Component 1  ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'Principal Component 2  ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.legend(title='Cluster', fontsize=11)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 2.7 Evaluation and Discussion

In [ ]:
print("""
EVALUATION AND DISCUSSION (2.7):
---------------------------------
Scatter Plot Evaluation:
  The PCA scatter plot reduces the 7-dimensional water quality dataset to
  2 principal components, making it possible to visualise the cluster
  structure in a 2D plot. The two components together explain the percentage
  of total variance printed above.

Cluster Visibility:
  The clusters identified by K-Means are visible and distinguishable in the
  PCA scatter plot. Some clusters are well-separated along PC1, which
  captures the largest proportion of variance and is most strongly influenced
  by features such as Water Temperature, Air Temperature and Dissolved Oxygen.
  This suggests that temperature-related measurements are the primary drivers
  of cluster separation — a physically meaningful result, as seasonal and
  geographic temperature differences often govern broader water quality patterns.

Effectiveness of PCA:
  PCA is effective for this dataset in the sense that meaningful cluster
  structure is still visible after reducing from 7 to 2 dimensions. However,
  some overlap between adjacent clusters is present, which is expected since
  the remaining variance (not captured by PC1 and PC2) contains additional
  discriminating information that is lost in the 2D projection. Using 3
  principal components would improve separability at the cost of requiring a
  3D visualisation. Overall, PCA provides a practical and interpretable
  dimensionality reduction that confirms the validity of the K-Means clustering
  solution on this water quality dataset.
""")